# 🤖 Laboratório de IA em Python — Aula 2 (prática no Colab)

**Informática para Internet · 1º ano · Introdução à Inteligência Artificial**

Hoje você deixa de **usar** IA e passa a **construir** uma pequena IA com as próprias mãos. Sem medo: o código já vem quase pronto, e você só precisa **mudar, testar e observar**.

### Como funciona
1. Para rodar uma célula de código: clique nela e aperte **Shift + Enter** (ou o botão ▶).
2. Rode as células **na ordem**, de cima para baixo.
3. Procure os blocos **✏️ Sua vez** — é lá que você mexe.
4. Os blocos **📝 Copie no caderno** são para escrever à mão.
5. Cada missão vale **XP**. Veja seu placar na última célula!

> ⚠️ Não coloque dados pessoais (nome completo, telefone, endereço) em nenhum código ou site de IA.

| Missão | O que você vai fazer |
|---|---|
| 0 | Preparar o laboratório e se apresentar |
| 1 | Ensinar um computador a diferenciar **gatos** e **cachorros** (aprendizado de máquina) |
| 2 | Criar um **detector de sentimentos** e descobrir onde ele erra |
| 3 | Construir um **gerador de texto** (mini IA generativa) |
| 4 | Usar o ChatGPT como **professor de programação** |

In [ ]:
# ▶ Rode esta célula primeiro (prepara o laboratório)
import random
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

XP = {}
def ganhar(chave, pontos, mensagem=""):
    """Dá XP uma única vez por desafio."""
    if chave not in XP:
        XP[chave] = pontos
        print(f"🎉 +{pontos} XP! {mensagem}")
    else:
        print(f"✅ Desafio já concluído ({XP[chave]} XP).")

def placar():
    total = sum(XP.values())
    niveis = [(0, "🐣 Calouro"), (20, "🔍 Aprendiz"), (45, "🧪 Cientista de Dados"), (70, "🚀 Mestre da IA")]
    nivel = [n for p, n in niveis if total >= p][-1]
    print(f"⭐ Você tem {total} XP — nível: {nivel}")
    print("Desafios concluídos:", ", ".join(XP) if XP else "nenhum ainda")

print("Laboratório pronto! 🧪")

---
## 🎮 Missão 0 — Olá, Python!  *(5 XP)*

Python é uma linguagem de programação muito usada em IA. Um programa é uma lista de instruções. Duas ideias bastam por enquanto:

- **variável**: uma "caixinha" com nome que guarda um valor → `idade = 15`
- **print()**: faz o computador escrever na tela

### ✏️ Sua vez
Troque `"Seu apelido aqui"` pelo seu apelido (mantenha as aspas) e rode a célula.

In [ ]:
apelido = "Seu apelido aqui"     # ✏️ troque o texto entre aspas
cor_favorita = "azul"             # ✏️ troque também

print("Olá,", apelido, "! Sua cor favorita é", cor_favorita)

if apelido != "Seu apelido aqui":
    ganhar("missao0", 5, "Primeiro programa feito!")
else:
    print("👀 Falta trocar o apelido na linha 1.")

---
## 🎮 Missão 1 — Ensinando o computador a ver a diferença  *(até 30 XP)*

Na aula, vimos que **aprendizado de máquina** é aprender com **exemplos**, não com regras escritas à mão.

Aqui temos **80 animais** com duas medidas cada: **peso** (kg) e **comprimento do focinho** (cm). Cada animal tem uma **etiqueta** (gato ou cachorro). O computador vai olhar os exemplos e tentar **adivinhar** a etiqueta de animais novos.

### Passo 1 — Conhecer os dados

In [ ]:
random.seed(7)
animais = []
for _ in range(40):
    animais.append([round(random.uniform(2.5, 6.5), 1), round(random.uniform(3.0, 5.4), 1), "gato"])
for _ in range(32):      # cachorros de porte médio e grande
    animais.append([round(random.uniform(8.0, 35.0), 1), round(random.uniform(6.0, 12.0), 1), "cachorro"])
for _ in range(8):       # cachorros pequenos: parecidos com gatos!
    animais.append([round(random.uniform(5.0, 9.0), 1), round(random.uniform(5.0, 7.0), 1), "cachorro"])
random.shuffle(animais)

treino = animais[:56]   # exemplos para APRENDER
teste  = animais[56:]   # animais "escondidos" para PROVAR que aprendeu

print("Primeiros 5 exemplos (peso, focinho, etiqueta):")
for a in treino[:5]:
    print(a)

def grafico(modelo_pontos=None):
    plt.figure(figsize=(7, 4.5))
    for etiqueta, cor, marca in [("gato", "tab:orange", "o"), ("cachorro", "tab:blue", "s")]:
        xs = [a[0] for a in treino if a[2] == etiqueta]
        ys = [a[1] for a in treino if a[2] == etiqueta]
        plt.scatter(xs, ys, c=cor, marker=marca, label=etiqueta, alpha=0.75)
    if modelo_pontos:
        for (x, y, rotulo) in modelo_pontos:
            plt.scatter([x], [y], c="red", marker="*", s=250, zorder=5)
            plt.annotate(rotulo, (x, y), textcoords="offset points", xytext=(8, 8))
    plt.xlabel("peso (kg)")
    plt.ylabel("comprimento do focinho (cm)")
    plt.title("Nossos exemplos de treino")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

grafico()

👀 **Observe o gráfico.** Os gatos (laranja) ficam juntos, no canto inferior esquerdo. Os cachorros (azul) se espalham — mas há **cachorros pequenos** no meio dos gatos! Em alguns lugares, **as cores se misturam** — é aí que a IA vai ter dificuldade.

### Passo 2 — Treinar e testar
O **k** diz quantos "vizinhos" o computador consulta. Para classificar um animal novo, ele olha os **k exemplos mais parecidos** e vota.

### ✏️ Sua vez
Mude o valor de `k` (tente 1, 3, 5, 9, 15) e rode várias vezes. Anote qual k deu a melhor **precisão**.

In [ ]:
k = 3        # ✏️ tente outros valores: 1, 3, 5, 9, 15

def treinar(exemplos, k):
    modelo = KNeighborsClassifier(n_neighbors=k)
    modelo.fit([[a[0], a[1]] for a in exemplos], [a[2] for a in exemplos])
    return modelo

def precisao(modelo):
    acertos = 0
    for a in teste:
        if modelo.predict([[a[0], a[1]]])[0] == a[2]:
            acertos += 1
    return acertos / len(teste)

modelo = treinar(treino, k)
p = precisao(modelo)
print(f"Com k={k}, a IA acertou {p*100:.0f}% dos animais do teste.")
if p >= 0.75:
    ganhar("m1_treino", 8, "Seu modelo aprendeu bem!")

### Passo 3 — Mais exemplos = IA mais esperta?
Vamos treinar a IA com **poucos exemplos** e depois com **muitos**, e comparar.

### ✏️ Sua vez
Troque a lista `quantidades` por outros números (entre 2 e 56) e veja o que acontece.

In [ ]:
quantidades = [2, 5, 10, 20, 56]    # ✏️ mude os números

resultados = []
for n in quantidades:
    m = treinar(treino[:n], k=1 if n < 3 else 3)
    resultados.append(precisao(m) * 100)

plt.figure(figsize=(6, 3.5))
plt.plot(quantidades, resultados, marker="o")
plt.xlabel("quantidade de exemplos de treino")
plt.ylabel("precisão (%)")
plt.title("Mais exemplos, mais acertos?")
plt.ylim(0, 105)
plt.grid(alpha=0.3)
plt.show()

for n, r in zip(quantidades, resultados):
    print(f"{n:>3} exemplos -> {r:.0f}% de acertos")

if len(quantidades) >= 4:
    ganhar("m1_dados", 6, "Você descobriu o papel dos dados!")

### Passo 4 — Teste a sua IA com um animal inventado
Escolha o **peso** e o **focinho** de um bicho (ex.: um gato gordo, um cachorro mini…) e veja o palpite.

### ✏️ Sua vez
Crie **3 animais** diferentes. Tente achar um que **engane** a IA!

In [ ]:
meus_animais = [
    (4.5, 4.0),      # ✏️ (peso, focinho) — invente!
    (30.0, 11.0),    # ✏️
    (6.0, 5.0),      # ✏️ tente achar um animal difícil
]

modelo = treinar(treino, k=3)
pontos = []
for peso, focinho in meus_animais:
    palpite = modelo.predict([[peso, focinho]])[0]
    prob = modelo.predict_proba([[peso, focinho]])[0].max()
    print(f"Peso {peso} kg, focinho {focinho} cm -> IA acha que é {palpite.upper()} ({prob*100:.0f}% de certeza)")
    pontos.append((peso, focinho, palpite))
grafico(pontos)

if len(meus_animais) >= 3 and meus_animais != [(4.5, 4.0), (30.0, 11.0), (6.0, 5.0)]:
    ganhar("m1_testes", 6, "Você testou a IA com seus próprios casos!")
else:
    print("👀 Troque os números dos animais para ganhar o XP.")

### 🧠 Desafio de reflexão  *(+10 XP — preencha a variável)*
A IA **errou** em algum caso? Por que você acha que ela errou? Responda na variável abaixo (mínimo 40 letras).

In [ ]:
minha_explicacao = ""     # ✏️ escreva entre as aspas

if len(minha_explicacao) >= 40:
    ganhar("m1_reflexao", 10, "Pensamento crítico ativado!")
else:
    print("Escreva uma explicação com pelo menos 40 caracteres.")

### 📝 Copie no caderno — *Aprendizado de máquina*
> **Aprendizado de máquina** = o computador aprende **padrões a partir de exemplos (dados)**, em vez de seguir regras escritas por uma pessoa.
> - **Treino**: exemplos para aprender · **Teste**: exemplos novos para conferir
> - **Mais dados bons → geralmente mais acertos**
> - Casos **parecidos demais** (cores misturadas no gráfico) geram **erros**.
> - Uma IA dá um **palpite com certeza (%)**, não uma verdade absoluta.

---
## 🎮 Missão 2 — Detector de sentimentos  *(até 30 XP)*

**Processamento de Linguagem Natural (PLN)** é ensinar computadores a lidar com textos. Vamos criar um detector que diz se uma frase é **positiva** ou **negativa** contando palavras.

### Passo 1 — Rodar o detector

In [ ]:
positivas = ["amei", "gostei", "legal", "ótimo", "bom", "incrível", "adorei", "feliz"]
negativas = ["odiei", "ruim", "péssimo", "horrível", "chato", "triste", "detestei"]

def analisar(frase):
    palavras = frase.lower().replace(",", "").replace(".", "").replace("!", "").split()
    pontos = 0
    for p in palavras:
        if p in positivas:
            pontos += 1
        if p in negativas:
            pontos -= 1
    return pontos

def mostrar(frase, funcao=analisar):
    pontos = funcao(frase)
    humor = "😀 POSITIVO" if pontos > 0 else "😠 NEGATIVO" if pontos < 0 else "😐 NEUTRO"
    print(f"{humor}  ({pontos:+d})  ->  {frase}")

mostrar("Eu amei essa aula, foi ótimo!")
mostrar("Que filme horrível, odiei")
mostrar("A aula é às treze horas")

### ✏️ Sua vez — Passo 2: enganar o detector
Escreva frases e descubra onde ele **erra**. Tente: frases com **"não"**, **ironia** ("Adorei acordar às 5h, né?") e palavras que não estão nas listas.

In [ ]:
minhas_frases = [
    "Eu não gostei do final",                 # ✏️ troque/acrescente frases
    "Adorei ficar duas horas na fila, né?",   # ✏️
    "Esse jogo é massa",                      # ✏️
]
erros_achados = 0
for f in minhas_frases:
    mostrar(f)

# Você marca quantos erros encontrou:
erros_achados = 0     # ✏️ conte quantas frases o detector julgou errado e escreva o número

if erros_achados >= 2:
    ganhar("m2_erros", 10, "Detetive de erros de IA!")
else:
    print("Ache pelo menos 2 frases que o detector errou e coloque o número em erros_achados.")

### ✏️ Sua vez — Passo 3: melhorar com novas palavras  *(+8 XP)*
Acrescente **pelo menos 3 palavras** novas às listas (gírias, por exemplo) e veja se o detector melhora.

In [ ]:
palavras_novas_positivas = []    # ✏️ ex.: ["massa", "top"]
palavras_novas_negativas = []    # ✏️ ex.: ["bosta", "lixo"]  (cuidado com palavrões!)

positivas += palavras_novas_positivas
negativas += palavras_novas_negativas

if len(palavras_novas_positivas) + len(palavras_novas_negativas) >= 3:
    ganhar("m2_palavras", 8, "Você ampliou o vocabulário da IA!")
else:
    print("Adicione pelo menos 3 palavras no total.")
mostrar("Esse jogo é massa")

### 🏆 Desafio chefe — Passo 4: ensinar a IA o "não"  *(+12 XP)*
O detector acha que *"não gostei"* é positivo. Corrija: **se a palavra anterior for "não", inverta o sinal** daquela palavra.

Dica de lógica: guarde a palavra anterior em uma variável enquanto percorre a frase.

Complete a função `analisar2` (o esqueleto já está pronto):

In [ ]:
def analisar2(frase):
    palavras = frase.lower().replace(",", "").replace(".", "").replace("!", "").split()
    pontos = 0
    anterior = ""
    for p in palavras:
        valor = 0
        if p in positivas:
            valor = 1
        if p in negativas:
            valor = -1
        # ✏️ COMPLETE: se a palavra anterior for "não", inverta o valor (valor = -valor)

        pontos += valor
        anterior = p
    return pontos

casos = [
    ("Eu gostei muito", 1),
    ("Eu não gostei", -1),
    ("Não odiei", 1),
    ("Eu não amei", -1),
    ("Odiei tudo", -1),
]
acertos = 0
for frase, esperado in casos:
    r = analisar2(frase)
    sinal = (r > 0) - (r < 0)
    ok = sinal == esperado
    acertos += ok
    print("✅" if ok else "❌", frase, "->", r)

if acertos == len(casos):
    ganhar("m2_chefe", 12, "Você ensinou a IA a entender negação!")
else:
    print(f"Passou em {acertos} de {len(casos)} casos. Continue tentando!")

<details>
<summary>💡 Dica (clique só se travar)</summary>

Dentro do `for`, antes de `pontos += valor`, escreva:

```python
if anterior == "não":
    valor = -valor
```
</details>

### 📝 Copie no caderno — *PLN*
> **PLN** = ensinar o computador a trabalhar com a linguagem humana.
> - Nosso detector **conta palavras**; não entende o **sentido**.
> - Ele erra com **negação** ("não gostei"), **ironia**, **gírias** e **contexto**.
> - Regras ajudam, mas a língua tem exceções demais → por isso hoje se usa **aprendizado de máquina** em PLN.

---
## 🎮 Missão 3 — Gerador de texto (mini IA generativa)  *(até 28 XP)*

O ChatGPT escolhe, uma de cada vez, **qual palavra vem depois da anterior**. Vamos construir uma versão minúscula dessa ideia: ele aprende **quais palavras aparecem depois de quais**, num texto, e depois **sorteia** o que vem a seguir.

### Passo 1 — Treinar com um texto

In [ ]:
texto = """
o gato dorme no sofá. o cachorro corre no parque. o gato come peixe.
o cachorro dorme no tapete. a aula de informática é legal.
a aula de programação é divertida. o aluno aprende a programar.
o aluno pergunta ao professor. o professor explica a aula.
a inteligência artificial aprende com exemplos.
"""

def aprender(texto):
    palavras = texto.lower().replace(".", " .").split()
    seguintes = {}
    for atual, proxima in zip(palavras, palavras[1:]):
        seguintes.setdefault(atual, []).append(proxima)
    return seguintes

def gerar(seguintes, inicio, tamanho=12):
    frase = [inicio]
    for _ in range(tamanho):
        opcoes = seguintes.get(frase[-1])
        if not opcoes:
            break
        frase.append(random.choice(opcoes))
        if frase[-1] == ".":
            break
    return " ".join(frase).replace(" .", ".")

seguintes = aprender(texto)
print("Depois de 'o', já apareceram:", seguintes["o"])
print("Depois de 'aula', já apareceram:", seguintes["aula"])

### ✏️ Sua vez — Passo 2: gerar frases
Rode **várias vezes** a célula abaixo: como há sorteio, **cada resposta é diferente** — igual ao ChatGPT! Troque a palavra inicial (`"o"`, `"a"`, `"aluno"`…).

In [ ]:
inicio = "o"     # ✏️ tente "a", "aluno", "gato"...

for _ in range(5):
    print(gerar(seguintes, inicio))

ganhar("m3_gerar", 4, "Primeiro texto gerado!")

### ✏️ Sua vez — Passo 3: seu próprio texto  *(+12 XP)*
Escreva **no mínimo 5 frases** sobre a sua escola, seus jogos ou seu dia (sem dados pessoais!). O gerador vai aprender o **seu estilo**.

In [ ]:
meu_texto = """
"""     # ✏️ escreva suas frases aqui, entre as aspas triplas (termine cada frase com ponto)

if meu_texto.count(".") >= 5:
    seguintes2 = aprender(meu_texto)
    inicio2 = meu_texto.lower().split()[0]
    print("Textos gerados:")
    for _ in range(5):
        print(" •", gerar(seguintes2, inicio2))
    ganhar("m3_meutexto", 12, "Sua IA aprendeu com o SEU texto!")
else:
    print("Escreva pelo menos 5 frases com ponto final.")

### 🧠 Descobrindo os limites  *(+12 XP)*
Se o texto de treino é **pequeno**, o gerador repete frases ou diz coisas sem sentido. O ChatGPT treina com uma quantidade **gigantesca** de textos — e ainda assim pode **inventar informações**.

Responda abaixo: *por que um gerador de texto pode escrever algo que parece certo, mas é falso?*

In [ ]:
resposta_limites = ""     # ✏️ escreva sua resposta (mínimo 50 caracteres)

if len(resposta_limites) >= 50:
    ganhar("m3_limites", 12, "Você entendeu o risco das 'alucinações'!")
else:
    print("Escreva ao menos 50 caracteres.")

### 📝 Copie no caderno — *IA generativa*
> **IA generativa** cria conteúdo novo (texto, imagem, som) **prevendo o que vem a seguir**, com base nos padrões que viu em muitos exemplos.
> - O resultado é sorteado → **cada resposta pode ser diferente**.
> - Ela **não "sabe" se é verdade**: só sabe o que é **provável**. Por isso pode **inventar** (*alucinação*).
> - Conclusão: **sempre conferir** informações importantes.

---
## 🎮 Missão 4 — O ChatGPT como professor de programação  *(até 20 XP)*

Agora você usa o ChatGPT **do jeito certo**: para **entender**, não para copiar tudo pronto.

### Parte A — Peça uma explicação  *(+8 XP)*
1. Copie o código da Missão 2 (`analisar`).
2. Abra o ChatGPT e cole este pedido junto com o código:

```
Sou aluno do 1º ano do ensino médio e estou aprendendo Python.
Explique, linha por linha e com palavras simples, o que este código faz.
Depois me proponha 2 melhorias que eu possa tentar sozinho.
```

3. Escreva abaixo **uma coisa que você entendeu melhor**:

In [ ]:
aprendi = ""     # ✏️ uma coisa nova que o ChatGPT explicou (mínimo 30 caracteres)

if len(aprendi) >= 30:
    ganhar("m4_explicacao", 8, "Você usou a IA para aprender!")
else:
    print("Escreva ao menos 30 caracteres.")

### Parte B — Teste o ChatGPT  *(+6 XP)*
Peça ao ChatGPT: *"Escreva uma função em Python que diga se uma frase é positiva ou negativa. Não use bibliotecas."*
Depois **cole o código dele na célula abaixo e rode**. Funcionou de primeira?

In [ ]:
# ✏️ Cole aqui o código que o ChatGPT gerou e rode para testar


codigo_colado = False   # ✏️ mude para True depois de colar e testar

if codigo_colado:
    ganhar("m4_teste", 6, "Código da IA testado — nunca confie sem testar!")

### Parte C — Avaliação final  *(+6 XP)*
Qual dessas afirmações é **verdadeira**? Escreva a letra em `resposta`.

- **A)** O código que o ChatGPT escreve está sempre correto.
- **B)** Devo testar e entender qualquer código gerado por IA antes de usar.
- **C)** A IA gerativa tem certeza do que fala.

In [ ]:
resposta = ""      # ✏️ "A", "B" ou "C"

if resposta.upper() == "B":
    ganhar("m4_quiz", 6, "Resposta certa!")
elif resposta:
    print("Quase! Releia a Missão 3 sobre alucinações.")

### 📝 Copie no caderno — *Receita para usar IA com sabedoria*
> 1. **Peça com contexto**: quem eu sou + o que quero + como quero a resposta.
> 2. **Entenda** antes de usar.
> 3. **Teste** (código) e **confira** (fatos) em fontes confiáveis.
> 4. **Nunca** envie dados pessoais.
> 5. IA é **ferramenta**: a responsabilidade pelo trabalho continua sendo **sua**.

---
## 🏁 Placar final
Rode a célula abaixo e mostre ao professor.

In [ ]:
placar()

### 🌟 Bônus (para quem terminou antes)
- Na Missão 1, adicione uma **terceira característica** (ex.: altura) e treine de novo.
- Na Missão 2, crie `analisar3` que também entenda **"muito"** (conta o dobro).
- Na Missão 3, treine o gerador com a letra de uma música **que você mesmo escreveu**.
- Peça ao ChatGPT para sugerir **um novo projeto de IA** que você consiga fazer com o que aprendeu hoje.